# 03 — Backend representation and distance verification
The approved `nonzero_fingerprints_v1` policy requires every saved query and index fingerprint to have at least one enabled bit before either backend uses real data. The runner scans the **full artifact**, then checks a deterministic 20-row fixture; this is not exhaustive retrieval. No graph or nearest-neighbor search is run.

Nonzero synthetic checks and representation preservation are required. Native zero-input probes are explicitly unsupported diagnostics, preserving observed NMSLIB NaN versus RDKit Tanimoto 0 for two zero vectors. No result is patched. See `docs/nonzero_fingerprint_policy.md`.

In [1]:
from pathlib import Path
import json, sys
from time import perf_counter_ns
import numpy as np
import nmslib
from rdkit import rdBase
from chemtellect.fingerprints import load_configuration
from chemtellect.fingerprint_store import verify_artifact
from chemtellect.nmslib_adapter import run_verification, VerificationFailure
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'src/chemtellect').is_dir()), None)
if ROOT is None: raise RuntimeError('Start JupyterLab within the repository.')
print('Repository:', ROOT, 'Interpreter:', sys.executable)
print({'Python': sys.version, 'NumPy': np.__version__, 'RDKit': rdBase.rdkitVersion, 'NMSLIB': nmslib.__version__})

Repository: /home/ruben/Coding Projects/chemtellect Interpreter: /home/ruben/Coding Projects/chemtellect/.venv/bin/python3
{'Python': '3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]', 'NumPy': '2.5.4', 'RDKit': '2026.09.1', 'NMSLIB': '2.1.2'}


Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib


## Select one existing artifact
Edit only the artifact path below to select another saved pool. Split linkage and output location are derived from artifact metadata and verified by the storage module. Packages stay separate; missing input stops without fallback or generation.

In [2]:
artifact_path = ROOT/'data/processed/chemicalspace_new/package_00/fingerprints/628c36671feeb38e1a97a34dd664bb54f43851374fcea4190a474479a12df427'

## Verify and load all saved rows
The manifest is initially read only to locate its linked split. `verify_artifact` validates that linkage, scientific identity, checksums, full array shapes and ordered record mapping before returning full arrays. No prefix is substituted for the saved index.

In [3]:
if not artifact_path.is_dir():
    raise FileNotFoundError(f'Select an existing fingerprint artifact: {artifact_path}. This notebook cannot generate it.')
start = perf_counter_ns()
selection_metadata = json.loads((artifact_path/'manifest.json').read_text())
split_path = ROOT/selection_metadata['split_path']
config = load_configuration(ROOT/'experiments/configs/morgan_r2_2048_chiral_v1.json')
metadata_loading_ns = perf_counter_ns() - start
start = perf_counter_ns()
manifest, arrays = verify_artifact(artifact_path, split_path, config, ROOT)
verification_loading_ns = perf_counter_ns() - start
records = json.loads((artifact_path/'records.json').read_text())
loading_timings = {'selection_metadata_loading': metadata_loading_ns,
                   'full_artifact_loading_and_integrity_verification': verification_loading_ns}
print(manifest['source'], manifest['package'], manifest['artifact_id'], manifest['split_id'])
print('Full shapes:', arrays['queries'].shape, arrays['index'].shape)

chemicalspace_new package_00 628c36671feeb38e1a97a34dd664bb54f43851374fcea4190a474479a12df427 f85552ea5b17e4829a09df82c79ad3bb2d762ee672c0f52de234dc7fc668e908
Full shapes: (100, 256) (1000, 256)


## Validate the full artifact and run compatibility checks
The reusable runner validates all rows before real reconstruction or registration. It then performs nonzero synthetic checks and all 210 unordered pairs including self among query/index rows 0–9. Unsupported zero-input diagnostics are separate. The full-artifact guard proves absence of zero rows; the small fixture checks distance compatibility. Both are required. Failures are saved before raising, with later checks marked not run.

In [4]:
try:
    report_path, report = run_verification(ROOT, manifest, arrays['queries'], arrays['index'], records, loading_timings)
except VerificationFailure as error:
    print('Report:', error.report_path)
    print('Validation:', error.report['nonzero_validation'])
    print(str(error))
    raise
print('Report:', report_path)
print('Overall passed:', report['passed'])
print('Full-artifact validation:', report['nonzero_validation'])

Report: /home/ruben/Coding Projects/chemtellect/experiments/results/backend_verification/chemicalspace_new/package_00/628c36671feeb38e1a97a34dd664bb54f43851374fcea4190a474479a12df427/run_6c6c717ff4554962a8b98de41b686ca9.json
Overall passed: True
Full-artifact validation: {'status': 'passed', 'summary': {'policy_identifier': 'nonzero_fingerprints_v1', 'query_rows_checked': 100, 'index_rows_checked': 1000, 'query_zero_rows': 0, 'index_zero_rows': 0}, 'affected_records': []}


## Inspect the supported checks and unsupported diagnostics
Report timings are diagnostic preparation measurements. Unique report filenames preserve prior runs, including the original unsuccessful report.

In [5]:
for name, check in report['supported_checks'].items():
    print(name, check['status'], 'comparisons:', check['comparisons'], 'max error:', check['maximum_absolute_error'])
print('Real fixture identities:', report['supported_checks']['real']['selected_records'])
print('Maximum supported-domain error:', report['maximum_absolute_error_supported_domain'])
print('Unsupported zero-zero observation:', report['unsupported_diagnostics']['pairs'][0])
print('Nonzero validation time (ns):', report['nonzero_validation_ns'])

synthetic passed comparisons: 21 max error: 3.405979698278827e-08
real passed comparisons: 210 max error: 3.628108813380493e-08
Real fixture identities: [{'temporary_id': 0, 'role': 'queries', 'row_index': 0, 'source_id': 'objectid:5bee9fbef127007b1bbcda24', 'line_number': 173}, {'temporary_id': 1, 'role': 'queries', 'row_index': 1, 'source_id': 'objectid:5be4ee58f127006b8486dbd8', 'line_number': 606}, {'temporary_id': 2, 'role': 'queries', 'row_index': 2, 'source_id': 'objectid:5be31b88f1270011af00f545', 'line_number': 1045}, {'temporary_id': 3, 'role': 'queries', 'row_index': 3, 'source_id': 'objectid:5be7ed1ef127003b0c644627', 'line_number': 862}, {'temporary_id': 4, 'role': 'queries', 'row_index': 4, 'source_id': 'objectid:5be0231af12700469a4c164c', 'line_number': 630}, {'temporary_id': 5, 'role': 'queries', 'row_index': 5, 'source_id': 'objectid:5bf134c5f127006fad94bc0e', 'line_number': 840}, {'temporary_id': 6, 'role': 'queries', 'row_index': 6, 'source_id': 'objectid:5bf14b87f12

## Boundary
Future exact-search and HNSW entry points must invoke the same guard on all selected query/index rows before preparing either backend or constructing a graph. Those entry points, exhaustive search, recall and performance benchmarking remain unimplemented.